# EdAcc Accent Fairness Study — Colab runner

Calls the same modules as the VSCode CLI, so both environments run identical code.

**Runtime → Change runtime type → T4 GPU** before starting.

---

### This notebook GATES. Read this once.

Three runs of this study produced invalid results that looked successful:

| What happened | Why it was invisible |
|---|---|
| Reference group filled with the wrong speakers | Group counts looked plausible |
| Whisper returned empty text for every utterance | WER computed as exactly 1.00, not an error |
| Code fixes never reached the runtime | Setup cell skipped extraction when the folder existed |

Each stage below now **verifies its own output and raises** if it is wrong.
When a cell prints `GATE FAILED`, stop. Do not run the next cell. The failure
is cheap now and expensive after two hours of GPU time.

Order: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14

In [ ]:
#@title 1. Setup — ALWAYS re-extracts (this is the fix for stale code)
import os, sys, glob, shutil, subprocess

ZIP  = "/content/edacc-fairness.zip"     # upload via the Files pane
REPO = "/content/edacc-fairness"

os.chdir("/content")
assert os.path.exists(ZIP), (
    f"{ZIP} not found. Upload edacc-fairness.zip using the Files pane "
    "(folder icon, left sidebar), then re-run this cell.")

# Unconditional overwrite. The previous version skipped extraction when the
# folder already existed, which silently kept old code across three sessions.
# outputs/ is not inside the archive, so results are never destroyed here.
subprocess.run(["unzip", "-qo", ZIP, "-d", "/content"], check=True)

for d in glob.glob("/content/edacc-fairness/**/__pycache__", recursive=True):
    shutil.rmtree(d, ignore_errors=True)
for m in [k for k in list(sys.modules) if k.startswith("src")]:
    del sys.modules[m]

os.chdir(REPO)
sys.path.insert(0, REPO)
print("extracted to", REPO)
print(sorted(os.listdir()))


In [ ]:
#@title 2. GATE — confirm the current code actually landed
import os, sys

problems = []

# audio_compat handles the datasets 4.x AudioDecoder return type.
if not os.path.exists("src/audio_compat.py"):
    problems.append("src/audio_compat.py missing (audio will fail to read)")

cfg = open("src/config.py").read()
if "ACCENT_TO_GROUP" not in cfg:
    problems.append("config.py has no ACCENT_TO_GROUP (old substring mapping)")
if '"america"' in cfg and "ACCENT_TO_GROUP" not in cfg:
    problems.append("config.py still contains the substring token lists")

mp = open("src/mapping.py").read()
if "canonical" not in mp:
    problems.append("mapping.py is the old substring version")

run = open("run.py").read()
if "--batch-size" not in run:
    problems.append("run.py has no --batch-size flag (old inference code)")
if "PYTORCH_CUDA_ALLOC_CONF" not in run:
    problems.append("run.py does not set the CUDA allocator config")

if problems:
    print("GATE FAILED — the runtime is running stale code:")
    for p in problems:
        print("   -", p)
    raise SystemExit("Re-upload edacc-fairness.zip, then re-run cell 1.")

print("GATE PASSED — current code is in place")

# Prove the mapping behaves correctly on the labels EdAcc actually contains.
for m in [k for k in list(sys.modules) if k.startswith("src")]:
    del sys.modules[m]
from src.mapping import match_group

expected = {
    "Mainstream US English": "us_baseline",   # the real US baseline
    "Latin American": None,                   # must NOT be us_baseline
    "Nigerian English": "nigerian",
    "Jamaican English": "jamaican",
    "Scottish English": "scottish",
    "Indian English": "indian",
    "Southern British English": None,         # present in EdAcc, not in the study
}
bad = {k: match_group(k) for k, v in expected.items() if match_group(k) != v}
for label, want in expected.items():
    got = match_group(label)
    print(f"  {'ok  ' if got == want else 'FAIL'}  {label:26} -> {got}")
if bad:
    raise SystemExit(f"GATE FAILED — mapping is wrong for: {list(bad)}")
print("\nmapping verified")


In [ ]:
#@title 3. Install dependencies
!pip install -q -r requirements.txt 2>&1 | tail -3
print("\nIf pip asks you to restart the runtime, do it, then re-run cells 1-3.")


In [ ]:
#@title 4. GPU check and memory budget
import torch, os

# Must be set before torch allocates. run.py sets this too; doing it here means
# any in-notebook torch use gets it as well.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if not torch.cuda.is_available():
    print("NO GPU DETECTED")
    print("Runtime -> Change runtime type -> T4 GPU, then re-run cells 1-4.")
else:
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  ({total:.1f} GB)")
    torch.cuda.empty_cache()
    free, _ = torch.cuda.mem_get_info()
    print(f"free now: {free/1e9:.1f} GB")

    # Whisper Large V3 is 1.55B params: ~3.1 GB in fp16, ~6.2 GB in fp32.
    # The dtype keyword was renamed torch_dtype -> dtype; passing the old name
    # can silently load full precision, which is what exhausted the T4 before.
    if total < 16:
        print("\nT4-class GPU. Use --batch-size 8. If OOM persists, drop to 4.")
        print("Batch size halves automatically on OOM and the run continues.")
    else:
        print("\nAmple memory. --batch-size 16 is safe.")


## Verification

Both cells below run in under a minute, need no GPU and no corpus. They confirm
the analysis is correct *before* any expensive work. The synthetic validation
injects known disparity ratios and checks the pipeline recovers them.

In [ ]:
#@title 5. Unit tests
r = get_ipython().getoutput("python -m pytest tests/ -q")
print("\n".join(r[-6:]))
if not any("passed" in line and "failed" not in line for line in r):
    raise SystemExit("GATE FAILED — tests did not pass. Stop here.")
print("\nGATE PASSED")


In [ ]:
#@title 6. End-to-end synthetic validation
r = get_ipython().getoutput("python validate_synthetic.py")
print("\n".join(r[-14:]))
if not any("PASS" in line for line in r):
    raise SystemExit("GATE FAILED — analysis chain did not recover known values.")
print("\nGATE PASSED")


## Corpus access

EdAcc is CC-BY-SA. Run the next cell only if the audit fails with a 401 or a
gated-dataset error.

In [ ]:
#@title 7. Hugging Face login (only if needed)
# from huggingface_hub import login
# login()


## Stage 1 — Audit

Enumerates every accent label EdAcc contains and counts utterances and speakers
per study group.

**The number that matters is `us_baseline`.** EdAcc holds 1,983 utterances
labelled `Mainstream US English`. If the audit reports roughly 172 with an
example label of `Latin American`, the old substring mapping is still active and
the reference group is contaminated. The gate in cell 9 checks this.

In [ ]:
#@title 8. Audit
!python run.py audit


In [ ]:
#@title 9. GATE — is the reference group correct?
import pandas as pd

audit = pd.read_csv("outputs/group_audit.csv")
print(audit.to_string(index=False))

row = audit[audit.accent_group == "us_baseline"]
if row.empty:
    raise SystemExit("GATE FAILED — no us_baseline row.")

n = int(row.iloc[0]["utterances"])
example = str(row.iloc[0].get("example_labels", ""))

print(f"\nus_baseline: {n} utterances")
if "Latin American" in example or n < 1000:
    print(f"example labels: {example}")
    raise SystemExit(
        "GATE FAILED — the reference group is wrong.\n"
        f"Expected roughly 1983 utterances of 'Mainstream US English', got {n}.\n"
        "The old substring mapping is still active. Re-upload the zip and "
        "re-run cells 1 and 2.")

for _, r in audit.iterrows():
    if r.accent_group != "(labelled but unmapped)" and not r.admitted_to_modelling:
        print(f"  ! {r.accent_group}: below viability threshold "
              f"({r.utterances} utts, {r.speakers} speakers) — descriptive only")

print("\nGATE PASSED — reference group is sound")


In [ ]:
#@title 10. Inspect the label distribution
import pandas as pd
raw = pd.read_csv("outputs/raw_accent_labels.csv")

acc = raw[raw.field == "accent"].sort_values("count", ascending=False)
print(f"{acc['value'].nunique()} distinct accent labels in EdAcc\n")
display(acc.head(35))

un = raw[raw.field == "unmapped"].sort_values("count", ascending=False)
if not un.empty:
    print("\nLargest groups NOT in the study (excluded by design):")
    display(un.head(12))


### Widening the design (optional)

EdAcc contains other L1 English varieties the study does not currently use:
Southern British English (1,371), Irish English (1,317), Kenyan English (1,157),
Ghanaian English (357), South African English (246).

Southern British and Irish English are **inner-circle native varieties**. Adding
either would strengthen the nativeness versus distributional-distance contrast,
which currently rests on Scottish English alone as its single native comparison
point. To add one, insert it into `ACCENT_TO_GROUP` in `src/config.py` and re-run
from cell 8.

This is a design decision, not a fix. Discuss it before changing it.

## Stage 2 — Manifest

One row per mapped utterance with speaker, gender, transcript and duration.

EdAcc has **no age column**, so the intersectional analysis is gender-only. The
paper must state this rather than promise an age breakdown.

In [ ]:
#@title 11. Build manifest
!python run.py manifest


## Stage 3 — Inference

### Why this stage failed before

Whisper Large V3 exhausted the T4 and every affected utterance was recorded as
empty. Empty hypotheses score as complete deletions, producing a WER of exactly
1.00 that looks like a result rather than a failure. Worse, the resume logic
treated those blank entries as already done, so re-running skipped them.

Four changes now apply:

1. Half precision is requested through both `dtype` and `torch_dtype`, since the
   keyword was renamed and passing the wrong one silently loads fp32.
2. Batch size halves on out-of-memory and retries, instead of collapsing.
3. Chunking is enabled only if some utterance exceeds 30 seconds.
4. The allocator uses expandable segments to limit fragmentation.

**Run the smoke test and its gate before the full pass.**

In [ ]:
#@title 12a. Free GPU memory before starting
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"free {free/1e9:.1f} / {total/1e9:.1f} GB")


In [ ]:
#@title 12b. Smoke test — Whisper on 50 utterances
import os
# Start clean so the gate below tests fresh output, not stale blanks.
if os.path.exists("outputs/hyp_whisper.csv"):
    os.remove("outputs/hyp_whisper.csv")
    print("removed previous whisper hypotheses")

!python run.py infer --system whisper --batch-size 8 --smoke 50


In [ ]:
#@title 12c. GATE — is Whisper actually producing text?
import pandas as pd

h = pd.read_csv("outputs/hyp_whisper.csv")
text = h["whisper_hyp"].fillna("").astype(str).str.strip()
blank = text.eq("").sum()
pct = 100 * blank / len(h)

print(f"{blank} blank of {len(h)} ({pct:.0f}%)\n")
print("sample hypotheses:")
for s in text[text != ""].head(3):
    print("   ", s[:100])

if pct > 20:
    raise SystemExit(
        f"GATE FAILED — {pct:.0f}% of hypotheses are empty.\n"
        "This is the failure that invalidated three previous runs. Empty text "
        "scores as complete deletion and yields WER 1.00.\n"
        "Delete outputs/hyp_whisper.csv, lower --batch-size to 4, and retry. "
        "Check the log above for 'pipeline built with dtype=float16'.")

print(f"\nGATE PASSED — Whisper is transcribing")


In [ ]:
#@title 13a. Full inference — Whisper Large V3 (1-3 h)
# Resumes from the 50 smoke utterances. Re-run this cell if the runtime drops;
# it continues from the checkpoint rather than restarting.
!python run.py infer --system whisper --batch-size 8


In [ ]:
#@title 13b. Check Whisper completeness
import pandas as pd
m = pd.read_csv("outputs/manifest.csv")
h = pd.read_csv("outputs/hyp_whisper.csv")
blank = h["whisper_hyp"].fillna("").astype(str).str.strip().eq("").sum()
print(f"transcribed {len(h)} of {len(m)} manifest rows")
print(f"blank: {blank} ({100*blank/len(h):.1f}%)")
if len(h) < len(m):
    print("\nIncomplete. Re-run cell 13a to resume.")
elif blank / len(h) > 0.05:
    print("\nMore than 5% blank. Those utterances will score as total deletions "
          "and bias their groups. Consider deleting and re-running at batch 4.")
else:
    print("\nComplete and clean.")


In [ ]:
#@title 13c. Full inference — wav2vec 2.0 (30-60 min)
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()      # Whisper's memory is freed with its process

!python run.py infer --system wav2vec2 --batch-size 8


In [ ]:
#@title 13d. Check wav2vec 2.0 completeness
import pandas as pd
m = pd.read_csv("outputs/manifest.csv")
h = pd.read_csv("outputs/hyp_wav2vec2.csv")
blank = h["w2v_hyp"].fillna("").astype(str).str.strip().eq("").sum()
print(f"transcribed {len(h)} of {len(m)}   blank: {blank} "
      f"({100*blank/len(h):.1f}%)")
print("\nsample:")
for s in h["w2v_hyp"].dropna().head(2):
    print("   ", str(s)[:100])


## Stage 4 — Analysis

Scores both systems under one shared normaliser, computes WER with speaker-level
bootstrap intervals, fits the Poisson disparity model with clustered standard
errors, runs the gender intersectional pass, and writes every numbered table.

In [ ]:
#@title 14. Analyse and emit tables
!python run.py analyse


In [ ]:
#@title 15. Display the paper tables
import pandas as pd, glob, os
for path in sorted(glob.glob("outputs/table_*.csv")):
    print("\n" + "=" * 72)
    print(os.path.basename(path))
    print("=" * 72)
    display(pd.read_csv(path))


In [ ]:
#@title 16. Sanity check on the results
import pandas as pd

t2 = pd.read_csv("outputs/table_v_2.csv")
print(t2.to_string(index=False))

warn = []
if (t2["WER"] == 1.0).any():
    warn.append("A WER of exactly 1.000 means empty hypotheses, not a finding.")
if (t2["speakers"] < 2).any():
    warn.append("A group with one speaker cannot have a bootstrap interval.")
if t2["WER"].isna().any():
    warn.append("Missing WER values present.")

if warn:
    print("\nWARNINGS:")
    for w in warn:
        print("   -", w)
else:
    print("\nNo structural problems detected. Results are ready to interpret.")


In [ ]:
#@title 17. Download everything
import shutil
from google.colab import files
shutil.make_archive("/content/edacc_results", "zip", "outputs")
files.download("/content/edacc_results.zip")


## If something goes wrong

**Reset the code but keep results:** re-run cells 1 and 2.

**Reset everything and start clean:**

```python
import shutil, os
shutil.rmtree("/content/edacc-fairness", ignore_errors=True)
# then re-run from cell 1
```

**Whisper still runs out of memory:** lower to `--batch-size 4`, then `2`. The
run gets slower but does not fail. Confirm the log shows
`pipeline built with dtype=float16`; if it shows the full-precision fallback
warning, that is the real problem.

**A gate fails:** send the printed output rather than working around it. Every
gate here exists because that specific failure has already happened once and was
invisible in the results.

---

`outputs/paper_tables.md` holds every table in markdown, ready to paste into
Section IV. The mapping, normaliser description and run configuration are
written alongside for the appendices.